In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk


### Configurations and Tools

In [2]:
root_path = Path("/Users/furkan/projects/mebar-radiology")
manifest_path = root_path / "data" / "manifest.csv"
output_csv = root_path / "preprocessing" / "configurations" / "no_alignment" / "notebooks" / "adc_scale_fix_metrics.csv"

# thresholds from the plan
SMALL_P99_THRESHOLD = 10.0
SCALE_FACTOR = 1000.0
BOUND_1819 = 1818.855
BOUND_3000 = 3000.0
CEILING_4095 = 4095.0
REL_TOL = 1e-4


In [3]:
from IPython.display import display, HTML


# render a dataframe fully (no pandas ellipsis) inside a scrollable container
def show_scrollable(df, max_height="300px"):
    display(HTML(f'<div style="max-height:{max_height}; overflow:auto; border:1px solid #999">{df.to_html()}</div>'))


# ADC scale fix exploration

In [4]:
manifest_df = pd.read_csv(manifest_path)
adc_df = manifest_df[manifest_df["modality"] == "ADC"].reset_index(drop=True)
adc_df["image_path"] = adc_df["image_path"].map(lambda p: str(root_path / p))
adc_df["mask_path"] = adc_df["mask_path"].map(lambda p: str(root_path / p))
adc_by_id = adc_df.set_index("sample_id")
adc_df.head()


,sample_id,batch_id,response_group,patient_name,patient_id,modality,patient_path,modality_path,nesting_depth,image_path,...,mask_spacing_y_mm,mask_spacing_z_mm,mask_origin,mask_space,shape_matches,spacing_matches,origin_matches,is_valid_sample,validation_errors,manifest_created_at
0,0d6f73a01539ae0d,batch-1,intermediate,"ABDULLAEVA, GULSARA",a28eea2bdc22b7bc,ADC,"data/raw/batch-1/INTERMEDIATE/ABDULLAEVA, GULSARA","data/raw/batch-1/INTERMEDIATE/ABDULLAEVA, GULS...",0,/Users/furkan/projects/mebar-radiology/data/ra...,...,1.785714,6.600000,"(-183.96600185888997,-168.05404737164,-244.827...",left-posterior-superior,True,True,True,True,NaN,2026-10-18T12:46:46.218195+00:00
1,c5574c732708c9c0,batch-1,intermediate,"AKKAYA, HATICE",c76cde132a53d60f,ADC,"data/raw/batch-1/INTERMEDIATE/AKKAYA, HATICE","data/raw/batch-1/INTERMEDIATE/AKKAYA, HATICE/ADC",0,/Users/furkan/projects/mebar-radiology/data/ra...,...,0.781250,3.600031,"(-26.884099999999997,10.567199999999977,-337.0...",left-posterior-superior,True,True,True,True,NaN,2026-10-18T12:46:46.218195+00:00
2,85e3e9f712fe9029,batch-1,intermediate,"ARSLAN, AYSEL",fd5ae5ecb097ff8e,ADC,"data/raw/batch-1/INTERMEDIATE/ARSLAN, AYSEL","data/raw/batch-1/INTERMEDIATE/ARSLAN, AYSEL/ADC",0,/Users/furkan/projects/mebar-radiology/data/ra...,...,1.785714,6.600000,"(-207.14045924210006,-168.23679757503001,-288....",left-posterior-superior,True,True,True,True,NaN,2026-10-18T12:46:46.218195+00:00
3,6d382c4f0f97ffa6,batch-1,intermediate,"BAZYLOV, ARYSTANBEK",6b49f8c65adbca6c,ADC,"data/raw/batch-1/INTERMEDIATE/BAZYLOV, ARYSTANBEK","data/raw/batch-1/INTERMEDIATE/BAZYLOV, ARYSTAN...",0,/Users/furkan/projects/mebar-radiology/data/ra...,...,0.819672,3.000019,"(-40.571000000000005,-35.280300000000025,-409....",left-posterior-superior,True,True,True,True,NaN,2026-10-18T12:46:46.218195+00:00
4,c650692054c62b07,batch-1,intermediate,"BOSTANCI, OMER",c6a6d3d00cdc624a,ADC,"data/raw/batch-1/INTERMEDIATE/BOSTANCI, OMER","data/raw/batch-1/INTERMEDIATE/BOSTANCI, OMER/ADC",0,/Users/furkan/projects/mebar-radiology/data/ra...,...,1.785714,6.600000,"(-213.96959732613007,-149.13160683988008,-343....",left-posterior-superior,True,True,True,True,NaN,2026-10-18T12:46:46.218195+00:00


## Helpers

In [5]:
def load_validated_sample(row):
    image = sitk.ReadImage(row["image_path"])
    mask = sitk.ReadImage(row["mask_path"])
    if image.GetSize() != mask.GetSize():
        return None

    image_vals = sitk.GetArrayFromImage(sitk.Cast(image, sitk.sitkFloat32)).astype(np.float64).ravel()
    mask_vals = sitk.GetArrayFromImage(mask).astype(np.float64).ravel()
    return image_vals, mask_vals


def finite_foreground(vals):
    # foreground = finite voxels that are neither the 0 nor the -256 background sentinel
    return np.isfinite(vals) & (vals != 0.0) & (vals != -256.0)


def ratio_equal(vals, target, rel_tol=REL_TOL):
    if vals.size == 0:
        return 0.0
    return float(np.count_nonzero(np.isclose(vals, target, rtol=rel_tol, atol=0.0)) / vals.size)


def ratio_gt_le(vals, lower, upper):
    if vals.size == 0:
        return 0.0
    return float(np.count_nonzero((vals > lower) & (vals <= upper)) / vals.size)


def ratio_gt(vals, threshold):
    if vals.size == 0:
        return 0.0
    return float(np.count_nonzero(vals > threshold) / vals.size)


## 2. Scale group assignment

In [6]:
mismatched = []
group_rows = []
small_rows = []
large_sample_ids = []

for _, row in adc_df.iterrows():
    loaded = load_validated_sample(row)
    if loaded is None:
        mismatched.append(row["sample_id"])
        continue

    image_vals, mask_vals = loaded
    fg = finite_foreground(image_vals)
    fg_vals = image_vals[fg]
    p99 = np.percentile(fg_vals, 99) if fg_vals.size else 0.0

    is_small = p99 < SMALL_P99_THRESHOLD
    scale_group = "small" if is_small else "large"
    group_rows.append({
        "sample_id": row["sample_id"],
        "scale_group": scale_group,
        "response_group": row["response_group"],
        "image_dtype": row["image_dtype"],
        "batch_id": row["batch_id"],
    })

    if is_small:
        values = image_vals * SCALE_FACTOR
        finite_vals = values[np.isfinite(values)]
        nonzero_finite = values[np.isfinite(values) & (values != 0.0)]
        tumor_vals = values[np.isfinite(values) & (mask_vals != 0.0)]

        smax = float(finite_vals.max()) if finite_vals.size else np.nan
        small_rows.append({
            "sample_id": row["sample_id"],
            "scale_group": scale_group,
            "response_group": row["response_group"],
            "image_dtype": row["image_dtype"],
            "batch_id": row["batch_id"],
            "small_max": smax,
            "small_saturated_ratio": ratio_equal(nonzero_finite, smax),
            "small_tumor_max": float(tumor_vals.max()) if tumor_vals.size else np.nan,
            "small_tumor_at_bound_ratio": ratio_equal(tumor_vals, smax),
        })
    else:
        large_sample_ids.append(row["sample_id"])

group_df = pd.DataFrame(group_rows)
small_df = pd.DataFrame(small_rows)

print(f"mismatched samples: {len(mismatched)} -> {mismatched}")
scale_counts = group_df["scale_group"].value_counts()
print(scale_counts)
assert scale_counts.get("small", 0) == 22 and scale_counts.get("large", 0) == 165, "unexpected scale group split"


mismatched samples: 1 -> ['2fa95f062483ba1b']
scale_group
large    165
small     22
Name: count, dtype: int64


## 3. Small group per-sample bounds

In [7]:
small_bound_min = float(np.min(small_df["small_max"]))
small_bound_median = float(np.median(small_df["small_max"]))

near_1819 = np.isclose(small_df["small_max"], BOUND_1819, rtol=REL_TOL, atol=0.0)
different = small_df[~near_1819][["sample_id", "small_max"]].reset_index(drop=True)

print(f"small samples near 1818.855: {int(near_1819.sum())} / {len(small_df)}")
print(f"small bound min / median: {small_bound_min:.4f} / {small_bound_median:.4f}")

show_scrollable(different)
show_scrollable(small_df)


small samples near 1818.855: 16 / 22
small bound min / median: 1623.6789 / 1818.8548


,sample_id,small_max
0,34457c052fa53b67,3367.249489
1,580736da8ab3d87a,2202.748775
2,61640251fdc38baa,1623.678923
3,9e594f345c74fecc,3641.424417
4,9d5fc7c15d1bd36e,4519.244194
5,c7a5674d74470e4b,5075.791359


,sample_id,scale_group,response_group,image_dtype,batch_id,small_max,small_saturated_ratio,small_tumor_max,small_tumor_at_bound_ratio
0,6da62ba89f121b65,small,intermediate,double,batch-1,1818.854809,1.059561e-01,1818.854809,0.004975
1,003da9c17d67938b,small,intermediate,double,batch-1,1818.854809,1.121652e-01,1525.821447,0.000000
2,f2ee2155e8541645,small,intermediate,double,batch-1,1818.854809,8.626543e-02,1466.994882,0.000000
3,34457c052fa53b67,small,intermediate,double,batch-1,3367.249489,3.616898e-06,1712.881327,0.000000
4,580736da8ab3d87a,small,nonresponder,double,batch-1,2202.748775,1.288224e-03,1408.253074,0.000000
5,61640251fdc38baa,small,nonresponder,double,batch-1,1623.678923,3.530196e-03,1623.678923,0.002425
6,9e594f345c74fecc,small,nonresponder,double,batch-1,3641.424417,3.875248e-06,2650.023699,0.000000
7,9672ecc7488b9389,small,nonresponder,double,batch-1,1818.854809,7.142857e-02,1818.854809,0.010259
8,c59b42edffd18083,small,nonresponder,double,batch-1,1818.854809,7.407683e-02,1818.854809,0.090426
9,9d5fc7c15d1bd36e,small,responder,double,batch-1,4519.244194,3.985969e-07,2653.268814,0.000000


## 4. Large group tumor distribution

In [8]:
large_rows = []
for sample_id in large_sample_ids:
    row = adc_by_id.loc[sample_id]
    image_vals, mask_vals = load_validated_sample(row)

    tumor_vals = image_vals[np.isfinite(image_vals) & (mask_vals != 0.0)]

    large_rows.append({
        "sample_id": sample_id,
        "scale_group": "large",
        "response_group": row["response_group"],
        "image_dtype": row["image_dtype"],
        "batch_id": row["batch_id"],
        "tumor_voxel_count": int(tumor_vals.size),
        "tumor_gt_1819_le_3000_ratio": ratio_gt_le(tumor_vals, BOUND_1819, BOUND_3000),
        "tumor_gt_3000_ratio": ratio_gt(tumor_vals, BOUND_3000),
        "tumor_eq_4095_ratio": ratio_equal(tumor_vals, CEILING_4095),
        "tumor_gt_min_small_bound_le_3000_ratio": ratio_gt_le(tumor_vals, small_bound_min, BOUND_3000),
        "tumor_gt_median_small_bound_le_3000_ratio": ratio_gt_le(tumor_vals, small_bound_median, BOUND_3000),
    })

large_df = pd.DataFrame(large_rows)
show_scrollable(large_df)


,sample_id,scale_group,response_group,image_dtype,batch_id,tumor_voxel_count,tumor_gt_1819_le_3000_ratio,tumor_gt_3000_ratio,tumor_eq_4095_ratio,tumor_gt_min_small_bound_le_3000_ratio,tumor_gt_median_small_bound_le_3000_ratio
0,0d6f73a01539ae0d,large,intermediate,unsigned short,batch-1,468,0.000000,0.000000,0.000000,0.000000,0.000000
1,c5574c732708c9c0,large,intermediate,unsigned short,batch-1,5473,0.005847,0.000000,0.000000,0.022291,0.005847
2,85e3e9f712fe9029,large,intermediate,unsigned short,batch-1,376,0.000000,0.000000,0.000000,0.000000,0.000000
3,6d382c4f0f97ffa6,large,intermediate,unsigned short,batch-1,1422,0.000000,0.000000,0.000000,0.000000,0.000000
4,c650692054c62b07,large,intermediate,unsigned short,batch-1,599,0.000000,0.000000,0.000000,0.000000,0.000000
5,ca2f17811a30d3a5,large,intermediate,unsigned short,batch-1,3161,0.000000,0.000000,0.000000,0.000316,0.000000
6,3280ab13a3838901,large,intermediate,unsigned short,batch-1,4179,0.011007,0.000000,0.000000,0.040201,0.011007
7,11adf6fc33900eb7,large,intermediate,unsigned short,batch-1,1008,0.180556,0.000000,0.000000,0.462302,0.180556
8,988742e3c62af03c,large,intermediate,unsigned short,batch-1,3853,0.001298,0.000000,0.000000,0.004153,0.001298
9,a30e3f0f275a24bb,large,intermediate,unsigned short,batch-1,2146,0.005126,0.000000,0.000000,0.036347,0.005126


## 5. Summary

In [9]:
def summarize(df, cols):
    return pd.DataFrame(
        {
            col: {
                "median": df[col].median(),
                "p90": df[col].quantile(0.90),
                "p95": df[col].quantile(0.95),
                "max": df[col].max(),
            }
            for col in cols
        }
    ).T


small_summary = summarize(
    small_df,
    ["small_max", "small_saturated_ratio", "small_tumor_max", "small_tumor_at_bound_ratio"],
)
large_summary = summarize(
    large_df,
    [
        "tumor_gt_1819_le_3000_ratio",
        "tumor_gt_3000_ratio",
        "tumor_eq_4095_ratio",
        "tumor_gt_min_small_bound_le_3000_ratio",
        "tumor_gt_median_small_bound_le_3000_ratio",
    ],
)

print("small summary")
display(small_summary)
print("large summary")
display(large_summary)


small summary


,median,p90,p95,max
small_max,1818.854809,3614.006925,4475.353205,5075.791359
small_saturated_ratio,0.066297,0.111544,0.113558,0.163933
small_tumor_max,1758.928716,1818.854809,2608.465254,2653.268814
small_tumor_at_bound_ratio,0.000000,0.015702,0.086717,0.107771


large summary


,median,p90,p95,max
tumor_gt_1819_le_3000_ratio,0.000489,0.261671,0.543851,0.837855
tumor_gt_3000_ratio,0.000000,0.000000,0.001940,0.231290
tumor_eq_4095_ratio,0.000000,0.000000,0.000000,0.002428
tumor_gt_min_small_bound_le_3000_ratio,0.004168,0.461688,0.700565,1.000000
tumor_gt_median_small_bound_le_3000_ratio,0.000489,0.261671,0.543851,0.837855


In [10]:
top10 = large_df.sort_values("tumor_gt_1819_le_3000_ratio", ascending=False).head(10)[
    ["sample_id", "tumor_gt_1819_le_3000_ratio"]
].reset_index(drop=True)

print("top 10 large samples by main metric")
show_scrollable(top10)

print("scale x response_group")
show_scrollable(pd.crosstab(group_df["scale_group"], group_df["response_group"]))

print("scale x image_dtype")
show_scrollable(pd.crosstab(group_df["scale_group"], group_df["image_dtype"]))

per_sample_df = pd.concat([small_df, large_df], ignore_index=True)
per_sample_df.to_csv(output_csv, index=False)
print(f"wrote {output_csv}")


top 10 large samples by main metric


,sample_id,tumor_gt_1819_le_3000_ratio
0,09dab1c2bb1fdfbf,0.837855
1,c79b2678007b3a37,0.810590
2,f6918a545e35a27d,0.762149
3,f69ef1ba6af8b7f2,0.746495
4,601209b5ac600c5d,0.705609
5,e1d90317f9501b9a,0.697420
6,b9fd91280cc5918f,0.636183
7,e3df20187504fe05,0.595297
8,8e6172fe747a9a15,0.555692
9,3f0c33701d8ba62a,0.496484


scale x response_group


response_group,intermediate,nonresponder,responder
scale_group,,,
large,45,37,83
small,6,7,9


scale x image_dtype


image_dtype,double,int,short,unsigned short
scale_group,,,,
large,0,3,2,160
small,22,0,0,0


wrote /Users/furkan/projects/mebar-radiology/preprocessing/configurations/no_alignment/notebooks/adc_scale_fix_metrics.csv
